# 07 - Bridging Loan LGD Model

**Exit-Driven Bridging LGD - Australian Bank-Style Portfolio Framework**

Bridging loan LGD is primarily an exit-risk problem. This notebook demonstrates a pragmatic, auditable framework around exit pathway uncertainty.

| Step | Description |
|---|---|
| 1 | Generate synthetic bridging loan default dataset |
| 2 | Build base LGD from exit drivers |
| 3 | Show delay effect on LGD |
| 4 | Apply failed-exit stress scenarios |
| 5 | Produce EAD-weighted outputs and exports |


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(49)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 90)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")



## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy assumptions are used where observed internal bridging workout tapes are unavailable.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status (standard policy):** this is a portfolio-project proxy baseline, **not** internally calibrated to real workout outcomes.
- **Output standard:** report `lgd_base`, `lgd_downturn`, `lgd_final`, EAD-weighted outputs, and base recovery-time metric (`exit_time_base`).


## Objective
Build an interview-ready bridging LGD module where severity is governed by exit risk and weighted outputs.

## Drivers
- Exit type (refinance vs sale)
- Exit certainty
- Valuation risk
- Time to exit and failed-exit probability

## Logic
- Base/downturn/final LGD with delay and failure overlays and weighted aggregation

## Downturn
- Exit delays and failed-refinance channels driving higher liquidation severity

## Validation
- Weighted scenario/segment/delay outputs and validation checks

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## Why Bridging Risk = Exit Risk

Bridging loans are short-tenor and rely on a credible exit. Default severity is driven by **how and when the exit occurs**:

- Exit type (`refinance` vs `sale`)
- Exit certainty (probability of successful execution)
- Valuation risk at exit
- Time to exit

Delay and failed exits usually convert refinance assumptions into forced-sale outcomes, raising LGD materially.


## 1. Generate Synthetic Bridging Loan Dataset


In [ ]:
n_cases = 230
rng = np.random.default_rng(49)

purposes = ['acquisition_bridge', 'construction_refi_bridge', 'residual_stock_exit_bridge']
purpose_weights = [0.40, 0.35, 0.25]

rows = []
for i in range(1, n_cases + 1):
    purpose = rng.choice(purposes, p=purpose_weights)

    gross_value = float(rng.uniform(2.5e6, 65e6))
    leverage = float(np.clip(rng.normal(0.72, 0.08), 0.45, 0.92))
    ead = gross_value * leverage * rng.uniform(0.98, 1.04)

    exit_type_initial = rng.choice(['refinance', 'sale'], p=[0.62, 0.38])
    exit_certainty = float(np.clip(rng.normal(0.63, 0.18), 0.05, 0.98))
    valuation_risk = float(np.clip(rng.normal(0.38, 0.20), 0.03, 0.95))
    time_to_exit_months = float(np.clip(rng.normal(8.0, 3.0), 2.0, 24.0))

    contract_rate_proxy = float(np.clip(rng.normal(0.082, 0.014), 0.050, 0.125))
    cost_of_funds_proxy = float(np.clip(rng.normal(0.060, 0.011), 0.035, 0.095))
    discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)

    failure_prob_base = float(np.clip(
        0.06
        + 0.50 * (1 - exit_certainty)
        + 0.26 * valuation_risk
        + 0.20 * ((time_to_exit_months - 6) / 18),
        0.01,
        0.95,
    ))
    failed_exit_flag = int(rng.uniform() < failure_prob_base)

    final_exit_type = 'sale' if (failed_exit_flag == 1 and exit_type_initial == 'refinance') else exit_type_initial
    time_to_exit_effective_months = float(np.clip(time_to_exit_months + failed_exit_flag * rng.uniform(4, 14), 2, 36))

    rows.append({
        'loan_id': f'BRG{i:04d}',
        'purpose': purpose,
        'ead': ead,
        'gross_value': gross_value,
        'exit_type_initial': exit_type_initial,
        'exit_certainty': exit_certainty,
        'valuation_risk': valuation_risk,
        'time_to_exit_months': time_to_exit_months,
        'time_to_exit_effective_months': time_to_exit_effective_months,
        'failure_prob_base': failure_prob_base,
        'failed_exit_flag': failed_exit_flag,
        'final_exit_type': final_exit_type,
        'contract_rate_proxy': contract_rate_proxy,
        'cost_of_funds_proxy': cost_of_funds_proxy,
        'discount_rate': discount_rate,
    })

bridge = pd.DataFrame(rows)

print('Bridging cases:', bridge.shape)
display(bridge['purpose'].value_counts())
display(bridge['final_exit_type'].value_counts())
bridge.head()



## 2. Base LGD from Exit Drivers


In [ ]:
def compute_bridging_lgd(df, delay_multiplier=1.0, valuation_addon=0.0, failure_addon=0.0, rate_addon=0.0):
    x = df.copy()

    exit_time = (x['time_to_exit_effective_months'] * delay_multiplier).clip(2, 42)
    valuation_risk = (x['valuation_risk'] + valuation_addon).clip(0.0, 1.0)
    fail_prob = (x['failure_prob_base'] + failure_addon).clip(0.0, 1.0)
    disc_rate = (x['discount_rate'] + rate_addon).clip(0.03, 0.20)

    is_sale = (x['final_exit_type'] == 'sale').astype(int)

    lgd_base = (
        0.10
        + 0.28 * (1 - x['exit_certainty'])
        + 0.22 * valuation_risk
        + 0.18 * ((exit_time - 4) / 20).clip(0, 1)
        + 0.10 * is_sale
        + 0.28 * fail_prob
    ).clip(0.03, 0.99)

    # Discounting penalty for longer exits
    discount_penalty = ((1 + disc_rate) ** (exit_time / 12.0) - 1).clip(0.0, 0.50)
    lgd = (lgd_base + 0.20 * discount_penalty).clip(0.03, 0.99)

    return lgd, exit_time

bridge['lgd_base'], bridge['exit_time_base'] = compute_bridging_lgd(bridge)

print(f"EAD-weighted base LGD: {exposure_weighted_average(bridge, 'lgd_base', 'ead'):.2%}")
display(bridge[['lgd_base', 'exit_time_base', 'exit_certainty', 'valuation_risk', 'failure_prob_base']].describe().round(4))



## 3. Delay Effect: Longer Exit -> Higher LGD


In [ ]:
bridge['exit_delay_bucket'] = pd.cut(
    bridge['exit_time_base'],
    bins=[0, 6, 9, 12, 18, 42],
    labels=['<=6m', '6-9m', '9-12m', '12-18m', '18m+'],
    right=True,
)

delay_rows = []
for b, grp in bridge.groupby('exit_delay_bucket', observed=True):
    delay_rows.append({
        'exit_delay_bucket': str(b),
        'loan_count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'ead'),
    })

delay_summary = pd.DataFrame(delay_rows).sort_values("ead_weighted_lgd_base").reset_index(drop=True)
display(delay_summary.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(delay_summary['exit_delay_bucket'], delay_summary['ead_weighted_lgd_base'], color='#dd8452')
ax.set_title('Delay Effect: Exit Time Bucket vs EAD-Weighted Base LGD')
ax.set_xlabel('Exit Time Bucket')
ax.set_ylabel('EAD-Weighted Base LGD')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'bridging_delay_vs_lgd.png'), dpi=140, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'bridging_delay_lgd_comparison.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
    print('Plot display skipped (set LGD_NOTEBOOK_SHOW_PLOTS=1 to show).')



## 4. Failed Exit Scenarios, Downturn LGD, and Final Overlay

Scenario design captures delay and failed-exit effects under stress.  
An explicit final overlay is then added to downturn LGD so cross-product comparison uses a consistent `lgd_final` output.


In [ ]:
scenario_specs = [
    {'scenario': 'base', 'delay_multiplier': 1.00, 'valuation_addon': 0.00, 'failure_addon': 0.00, 'rate_addon': 0.000},
    {'scenario': 'delay_stress', 'delay_multiplier': 1.30, 'valuation_addon': 0.02, 'failure_addon': 0.03, 'rate_addon': 0.005},
    {'scenario': 'failed_exit_stress', 'delay_multiplier': 1.45, 'valuation_addon': 0.04, 'failure_addon': 0.10, 'rate_addon': 0.010},
    {'scenario': 'combined_downturn', 'delay_multiplier': 1.60, 'valuation_addon': 0.07, 'failure_addon': 0.18, 'rate_addon': 0.020},
]

scenario_rows = []
for s in scenario_specs:
    lgd_s, exit_time_s = compute_bridging_lgd(
        bridge,
        delay_multiplier=s['delay_multiplier'],
        valuation_addon=s['valuation_addon'],
        failure_addon=s['failure_addon'],
        rate_addon=s['rate_addon'],
    )
    name = s['scenario']
    bridge[f'lgd_{name}'] = lgd_s
    bridge[f'exit_time_{name}'] = exit_time_s

    overlay_s = (
        0.010
        + 0.032 * bridge['failed_exit_flag']
        + 0.014 * bridge['valuation_risk']
        + 0.012 * ((exit_time_s - 9.0) / 18.0).clip(0.0, 1.0)
    ).clip(0.010, 0.090)
    bridge[f'final_overlay_addon_{name}'] = overlay_s
    bridge[f'lgd_final_{name}'] = (lgd_s + overlay_s).clip(0.0, 1.0)

    scenario_rows.append({
        'scenario': name,
        'ead_weighted_lgd_base': exposure_weighted_average(bridge, f'lgd_{name}', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(bridge, f'lgd_{name}', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(bridge, f'lgd_final_{name}', 'ead'),
        'mean_exit_time_months': float(exit_time_s.mean()),
    })

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary['mean_recovery_time_months'] = scenario_summary['mean_exit_time_months']
bridge['lgd_downturn'] = bridge['lgd_combined_downturn']
bridge['final_overlay_addon'] = bridge['final_overlay_addon_combined_downturn']
bridge['lgd_final'] = bridge['lgd_final_combined_downturn']

display(scenario_summary.round(4))

failed_vs_non = (
    bridge.groupby('failed_exit_flag', observed=True)
    .apply(
        lambda g: pd.Series({
            'loan_count': len(g),
            'total_ead': g['ead'].sum(),
            'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base', 'ead'),
            'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead'),
            'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final', 'ead'),
        }),
        include_groups=False,
    )
    .reset_index()
)
display(failed_vs_non.round(4))



## 5. Weighted Outputs, Validation, and Exports


In [ ]:
segment_rows = []
for (purpose, final_exit), grp in bridge.groupby(['purpose', 'final_exit_type'], observed=True):
    segment_rows.append({
        'purpose': purpose,
        'final_exit_type': final_exit,
        'loan_count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(grp, 'lgd_downturn', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(grp, 'lgd_final', 'ead'),
        'mean_exit_certainty': grp['exit_certainty'].mean(),
        'mean_valuation_risk': grp['valuation_risk'].mean(),
        'mean_exit_time_base': grp['exit_time_base'].mean(),
    })

segment_summary = pd.DataFrame(segment_rows).sort_values(
    ['ead_weighted_lgd_final', 'total_ead'], ascending=[False, False]
).reset_index(drop=True)
display(segment_summary.round(4))

validation_checks = pd.DataFrame([
    {'check_name': 'lgd_base_between_0_and_1', 'passed': bool(bridge['lgd_base'].between(0, 1).all())},
    {'check_name': 'lgd_downturn_between_0_and_1', 'passed': bool(bridge['lgd_downturn'].between(0, 1).all())},
    {'check_name': 'lgd_final_between_0_and_1', 'passed': bool(bridge['lgd_final'].between(0, 1).all())},
    {'check_name': 'downturn_not_below_base', 'passed': bool((bridge['lgd_downturn'] >= bridge['lgd_base']).all())},
    {'check_name': 'final_not_below_downturn', 'passed': bool((bridge['lgd_final'] >= bridge['lgd_downturn']).all())},
    {'check_name': 'delay_stress_has_longer_exit_time', 'passed': bool((bridge['exit_time_combined_downturn'] >= bridge['exit_time_base']).all())},
])
display(validation_checks)

bridge.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_segment_summary.csv'), index=False)
delay_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_delay_summary.csv'), index=False)
scenario_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_scenario_summary.csv'), index=False)
validation_checks.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_validation_checks.csv'), index=False)

print('Saved bridging outputs:')
print('- bridging_loan_level_output.csv')
print('- bridging_segment_summary.csv')
print('- bridging_delay_summary.csv')
print('- bridging_scenario_summary.csv')
print('- bridging_validation_checks.csv')



## Assumptions and Limitations

- Bridging portfolio data is synthetic and for demonstration only.
- Exit certainty, valuation risk, and failure probability are proxy variables in absence of internal workout tapes.
- Downturn scenarios are designed to transparently show how delayed/failed exits increase LGD.
